# Explore source data

Before building any transformation, look at what's actually sitting in Postgres: the trusted `generation_summary` table (Ireland, via ENTSO-E) and the raw `raw_ingestions` weather snapshots (Dublin, via Open-Meteo).

Uses the same `PostgresDatabase` class the rest of the pipeline does, so it reads from whichever database your environment is pointed at (local `docker-compose` by default, or production if you export the Neon env vars before launching Jupyter) - no separate connection logic to maintain here.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from load_db import PostgresDatabase

db = PostgresDatabase()

## Generation data (Ireland, `generation_summary`)

Validated, transformed, upserted - one row per 15-minute interval.

In [2]:
generation_df = db.query_generation_summary(country_code="IE", limit=50_000)
generation_df["timestamp"] = pd.to_datetime(generation_df["timestamp"], utc=True)

print(f"{len(generation_df)} rows")
if not generation_df.empty:
    print(f"{generation_df['timestamp'].min()}  to  {generation_df['timestamp'].max()}")
generation_df.sort_values("timestamp").head()

291 rows
2026-08-14 15:28:54.984643+00:00  to  2026-08-15 15:28:57.925777+00:00


/Users/stephenquirke/Library/CloudStorage/OneDrive-Personal/Documents/developer/energy-data-pipeline/src/load_db.py:251: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(


,id,timestamp,country_code,total_generation_mw,renewable_mw,renewable_pct,carbon_intensity_g_per_kwh,processed_at
290,1,2026-08-14 15:28:54.984643+00:00,IE,1450.0,550.0,37.93,270.3,2026-08-15 15:28:55.003096+00:00
289,98,2026-08-14 15:28:56.457829+00:00,IE,1450.0,550.0,37.93,270.3,2026-08-15 15:28:56.469703+00:00
288,195,2026-08-14 15:28:57.925777+00:00,IE,1450.0,550.0,37.93,270.3,2026-08-15 15:28:57.936591+00:00
287,2,2026-08-14 15:43:54.984643+00:00,IE,1482.0,567.0,38.26,267.8,2026-08-15 15:28:55.003096+00:00
286,99,2026-08-14 15:43:56.457829+00:00,IE,1482.0,567.0,38.26,267.8,2026-08-15 15:28:56.469703+00:00


In [3]:
generation_df[["total_generation_mw", "renewable_mw", "renewable_pct", "carbon_intensity_g_per_kwh"]].describe()

,total_generation_mw,renewable_mw,renewable_pct,carbon_intensity_g_per_kwh
count,291.000000,291.000000,291.000000,291.000000
mean,1840.020619,825.690722,44.422887,239.584536
std,214.124133,173.347251,4.556561,19.679804
min,1450.000000,550.000000,37.930000,212.000000
25%,1696.000000,679.000000,40.100000,217.400000
50%,1828.000000,795.000000,42.450000,248.900000
75%,1998.000000,979.000000,49.220000,256.200000
max,2288.000000,1153.000000,50.650000,270.300000


## Raw weather snapshots (`raw_ingestions`, source = weather)

Every scheduled pipeline run fetches a fresh rolling window from Open-Meteo and appends it as its own row - `raw_ingestions` has no `UNIQUE` constraint, by design, so this is the full history of every snapshot ever taken, overlaps and all. Querying directly with SQL here rather than `get_recent_raw()`, since that method's `limit` is meant for "give me the last N," not "give me everything."

In [4]:
with db.connection() as conn, conn.cursor() as cur:
    cur.execute(
        "SELECT ingested_at, payload FROM raw_ingestions WHERE source = %s ORDER BY id",
        ("weather",),
    )
    weather_snapshots = cur.fetchall()

print(f"{len(weather_snapshots)} weather ingestion snapshots")
if weather_snapshots:
    ingested_at, payload = weather_snapshots[-1]
    print(f"most recent snapshot: {ingested_at}, {len(payload)} hourly readings")
    payload[:3]

3 weather ingestion snapshots
most recent snapshot: 2026-08-15 15:28:57.940101+00:00, 25 hourly readings


## Note for the next notebook

Two things `02_transformations.ipynb` has to handle that this exploration surfaces:

1. **Overlapping snapshots** - the same hour appears in multiple weather snapshots (each run re-fetches `past_days` of history), so building a clean hourly series means deduping, not just concatenating.
2. **Different cadences** - `generation_summary` is 15-minute resolution, weather is hourly. They need to be aligned onto one shared cadence before anything can be correlated.